# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhaledObeid/KhaledObeid-flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: scoring / ranking — a learned probability consumed as an ordering, not a yes/no verdict.**

My lane (declining-content triage) hands an editor a *work list*, so the output that matters is the
order of the list under a capacity limit, not a per-page class. Three measured reasons the ranking
framing is the right one:

- **A binary answer is nearly free here.** 54.2% of pages in this slice are already flagged declining,
  and 13,152 of them still have measurable traffic. Telling an editor "these 13,152 pages are
  declining" changes nothing — they can only touch a handful this week.
- **Capacity is tiny next to the queue.** A team reviewing 20 pages a week gets through about 8% of
  that pool in a year. Only the very top of the ordering is ever read, so the metric and the model
  have to be about the top, not about overall accuracy.
- **The loss is heavy-tailed, so order carries almost all the value.** The top 5% of the pool by
  impressions lost (last 30 days vs the 30 before) holds 48.8% of all impressions the pool is losing.
  Getting the ordering right at the top is worth more than being right everywhere else.

So: train a classifier, but **use its probability as a score**, combine it with the traffic actually at
stake, and judge it with top-K metrics (section 3). Clustering is out — I am not looking for page
archetypes; regression on a continuous drop is possible later but the label I can defend first is a
yes/no outcome (section 2), so a calibrated classifier used as a ranker is the honest starting point.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import numpy as np
import pandas as pd

# make the notebook runnable from Colab or a local clone
while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# the candidate pool my lane actually ranks: flagged declining AND big enough to be worth an hour
declining = df["trend_direction"] == "down"
measurable = (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
pool = df[declining & measurable].copy()

backlog = pool.groupby("client_id").size()
print(f"candidate pool: {len(pool):,} pages across {pool['client_id'].nunique()} clients")
print(f"per-client backlog  median {int(backlog.median())}  75th pct {int(backlog.quantile(0.75))}  max {int(backlog.max())}")

for cap in (10, 20, 50):
    print(f"a team reviewing {cap:>2}/week reaches {cap * 52 / len(pool):.0%} of the pool in a year")

# how concentrated is the thing we are trying not to lose?
pool["impressions_lost_30d"] = (pool["impressions_prev_30d"] - pool["impressions_last_30d"]).clip(lower=0)
lost_sorted = pool["impressions_lost_30d"].sort_values(ascending=False)
total_lost = lost_sorted.sum()
print(f"\npool is losing {total_lost:,.0f} impressions per 30 days")
for q in (0.01, 0.05, 0.25):
    n = int(len(lost_sorted) * q)
    print(f"  top {q:>4.0%} of pages ({n:>4} pages) hold {lost_sorted.head(n).sum() / total_lost:.1%} of that loss")
print("\n-> the decision is 'what goes first', not 'is it declining' => scoring/ranking, judged at top-K")

candidate pool: 13,152 pages across 29 clients
per-client backlog  median 96  75th pct 521  max 3218
a team reviewing 10/week reaches 4% of the pool in a year
a team reviewing 20/week reaches 8% of the pool in a year
a team reviewing 50/week reaches 20% of the pool in a year

pool is losing 14,065,153 impressions per 30 days
  top   1% of pages ( 131 pages) hold 22.7% of that loss
  top   5% of pages ( 657 pages) hold 48.8% of that loss
  top  25% of pages (3288 pages) hold 83.9% of that loss

-> the decision is 'what goes first', not 'is it declining' => scoring/ranking, judged at top-K


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Two labels — one I have today and cannot trust, one I have to build.**

*What I have today (starter slice): `is_declining_label = (trend_direction == "down")` — a **defined
rule**, not an observed outcome.* Three things disqualify it as my lane's real target:

1. **It is arithmetic, not a finding.** `trend_direction` is a threshold on last-30d vs prev-30d
   impressions. Rebuilding it from those two columns reproduces all 30,000 rows exactly (100.000%
   agreement, cell below). A model that predicts it is re-deriving a subtraction I already have.
2. **It is circular for my lane.** My candidate pool is *already* the flagged-declining pages — the
   label is 1 on 100% of the rows I need to rank. It cannot order the queue.
3. **Its window overlaps the features.** Every trailing-90d total contains the last-30-day window that
   defines the label — a median 22% of a page's 90-day impressions sit inside the label window. So even
   "safe" features here partly contain the answer.

*The target I will actually predict (from the warehouse daily facts, weeks 4-6):* pick a decision date
`t` per client; **features come only from the 90 days ending at `t`**; the **label is observed after
`t`** — did the page keep losing impressions over `(t, t+30]` relative to the 30 days ending at `t`,
past a magnitude threshold and a minimum-volume floor I will write down in the data contract.

That flips the question from *"is this page declining right now"* (arithmetic) to **"will this page keep
bleeding if nobody touches it"** (worth an editor's hour). It is an observed outcome, it arrives 30 days
after the decision point, and its window never touches the feature window. The daily fact table
(78.8M rows, 2025-01-27 → 2026-06-30) is what makes that window possible; the starter CSV has no
post-snapshot column at all, which is exactly why the proxy is the only label available in it.

The priority score an editor sees will then be `P(keeps declining) × impressions at stake` — risk times
size — not the probability alone.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. the starter label is a rule: rebuild trend_direction from its two inputs
prev, last = df["impressions_prev_30d"], df["impressions_last_30d"]
pct = np.where(prev > 0, (last - prev) / prev.replace(0, np.nan) * 100, np.nan)
rebuilt = np.select(
    [(prev == 0) & (last > 0), (prev == 0) & (last == 0), pct > 20, pct < -20],
    ["new", "flat", "up", "down"],
    default="stable",
)
agreement = (rebuilt == df["trend_direction"].values).mean()
print(f"rebuilt trend_direction from 2 columns -> agrees on {agreement:.3%} of {len(df):,} rows")
print(f"label rate inside my candidate pool: {(pool['trend_direction'] == 'down').mean():.0%}  <- circular, cannot rank")

# 2. the label window sits INSIDE the feature window
overlap = (df["impressions_last_30d"] / df["impressions_90d"].replace(0, np.nan))
print(f"\nshare of a page's 90d impressions that fall in the label window: median {overlap.median():.0%}, mean {overlap.mean():.0%}")
print(f"pages that went dark in the label window (0 impressions in the last 30d): {(overlap == 0).sum():,}")

# 3. nothing in this file is measured AFTER the snapshot
windows = sorted(c for c in df.columns if "_90d" in c or "_30d" in c)
print(f"\ntime-window columns in the starter CSV ({len(windows)}): all trailing, none forward-looking")
print("  " + ", ".join(windows[:7]) + ", ...")
print("\n-> proxy label today; observed forward-window label (t, t+30] from the daily warehouse facts in week 4-5")

rebuilt trend_direction from 2 columns -> agrees on 100.000% of 30,000 rows
label rate inside my candidate pool: 100%  <- circular, cannot rank

share of a page's 90d impressions that fall in the label window: median 22%, mean 26%
pages that went dark in the label window (0 impressions in the last 30d): 2,547

time-window columns in the starter CSV (14): all trailing, none forward-looking
  ai_sessions_90d, clicks_90d, clicks_last_30d, clicks_prev_30d, engaged_sessions_90d, impressions_90d, impressions_last_30d, ...

-> proxy label today; observed forward-window label (t, t+30] from the daily warehouse facts in week 4-5


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: mean per-client precision@20 — of the 20 pages I put on one editor's desk for one
client, how many really were losing ground.** K = 20 is one editor-week, so the metric is measured
exactly where the decision happens. I average it across clients instead of pooling, because a pooled
top-20 is dominated by the two largest accounts.

**Secondary (decision value, not prediction quality): share of the pool's at-risk impressions captured
in the top K.** A queue can be 90% "correct" and still hold only tiny pages. The oracle ceiling here is
7.0% of lost impressions in 20 pages, 19.7% in 100 — that is the scale of what a perfect ranking could
protect.

**What "good" means, written down before any model exists:**

- The bar is the **base rate**, and it is client-specific: 0.598 pooled, but per-client it runs from
  0.15 to 0.96 (median 0.62). Beating 0.598 on a client whose base rate is 0.90 is not skill.
- The naive orderings I can build today **do not clear that bar**: traffic-first gets 0.473, stalest-first
  0.596, random 0.592 — against a 0.598 base rate. Ranking by traffic is *worse than random* here: big
  pages decline less often in this slice. That is a real result, and it is the honest bar to beat.
- **Noise band.** At one client with K=20 the 95% band on a proportion is ±0.215 — a single client's top-20
  proves nothing. Averaged over the 24 clients with ≥20 eligible pages (480 decisions) it tightens to
  ±0.044. So: **+10pp over the base-rate/rule bar = real; under +5pp = noise**, and I report a bootstrap
  interval, not a point estimate.
- I also keep pooled precision@50 for comparability with the starter pipeline (rules 0.240 → random
  forest 0.740 on this slice, client-holdout).

All of these are computed against the *proxy* label today. The metric machinery does not change when the
forward-window label lands in week 5 — only the label does.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(y, score, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="mergesort")[:k]
    return float(np.asarray(y)[order].mean())

# stand-in evaluation set while the forward label does not exist yet: all measurable pages
eval_pool = df[measurable].copy()
y = (eval_pool["trend_direction"] == "down").astype(int).values
print(f"eval rows {len(eval_pool):,}  pooled base rate {y.mean():.3f}")

per_client_base = eval_pool.groupby("client_id")["trend_direction"].apply(lambda s: (s == "down").mean())
sizes = eval_pool.groupby("client_id").size()
per_client_base = per_client_base[sizes >= 20]
print(f"per-client base rate: min {per_client_base.min():.2f}  median {per_client_base.median():.2f}  max {per_client_base.max():.2f}  ({len(per_client_base)} clients with >=20 eligible pages)")

# the bar: what do the orderings I can build WITHOUT a model achieve?
rng = np.random.default_rng(11)
rules = {
    "traffic first  (impressions_90d)": eval_pool["impressions_90d"].values,
    "stalest first  (days_since_last_update)": eval_pool["days_since_last_update"].values,
    "random order": rng.random(len(eval_pool)),
}
print("\nmean per-client precision@20 (K = one editor-week):")
for name, score in rules.items():
    vals = [precision_at_k(y[idx], np.asarray(score)[idx], 20)
            for idx in eval_pool.groupby("client_id").indices.values() if len(idx) >= 20]
    print(f"  {name:<40} {np.mean(vals):.3f}   (per-client sd {np.std(vals):.3f})")
print(f"  {'base rate (do nothing)':<40} {y.mean():.3f}")

# how much noise is in that number?
p = y.mean()
for k, label in [(20, "one client, K=20"), (480, "24 clients x 20 = 480 decisions")]:
    print(f"\n95% noise band, {label}: +/- {1.96 * np.sqrt(p * (1 - p) / k):.3f}")

# secondary metric: the ceiling on impressions protected
lost = pool["impressions_lost_30d"].sort_values(ascending=False)
print("\nperfect-ranking ceiling on at-risk impressions captured:")
for k in (20, 50, 100):
    print(f"  top {k:>3}: {lost.head(k).sum() / lost.sum():.1%} of the pool's 30-day impression loss")

eval rows 22,006  pooled base rate 0.598
per-client base rate: min 0.15  median 0.62  max 0.96  (24 clients with >=20 eligible pages)

mean per-client precision@20 (K = one editor-week):
  traffic first  (impressions_90d)         0.473   (per-client sd 0.235)
  stalest first  (days_since_last_update)  0.596   (per-client sd 0.214)
  random order                             0.592   (per-client sd 0.229)
  base rate (do nothing)                   0.598

95% noise band, one client, K=20: +/- 0.215

95% noise band, 24 clients x 20 = 480 decisions: +/- 0.044

perfect-ranking ceiling on at-risk impressions captured:
  top  20: 7.0% of the pool's 30-day impression loss
  top  50: 13.2% of the pool's 30-day impression loss
  top 100: 19.7% of the pool's 30-day impression loss


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item, for one client, over one trailing 90-day snapshot.** Verified below:
30,000 rows, `content_id` unique, zero duplicates on the (`client_id`, `content_id`) grain — so the
grain is real, not assumed.

The unit I *model* is narrower than the file. The funnel:

| Step | Rows | Why |
|---|---:|---|
| starter slice | 30,000 | one page per row |
| measurable traffic (≥100 impressions/90d, ≥1 session) | 22,006 | below this, a 30-day move is noise, not decline |
| + flagged declining | 13,152 | the queue an editor actually faces (29 of 32 clients) |

`content_id` and `client_id` are pseudonyms — **grouping and splitting only, never features**. `client_id`
is what train/test splits are grouped on, so the model is tested on accounts it has never seen.

In the warehouse the grain gains a time axis: **one content item × one decision date**, which is what
makes the forward-window label of section 2 expressible. Same row meaning, plus "as of when".


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"rows {len(df):,} | content_id unique: {df['content_id'].is_unique} | "
      f"duplicate (client_id, content_id) pairs: {int(df.groupby(['client_id', 'content_id']).size().gt(1).sum())}")

funnel = pd.DataFrame(
    [
        ("starter slice", len(df), df["client_id"].nunique()),
        ("measurable traffic", int(measurable.sum()), df.loc[measurable, "client_id"].nunique()),
        ("+ flagged declining (my pool)", len(pool), pool["client_id"].nunique()),
    ],
    columns=["step", "rows", "clients"],
)
print()
print(funnel.to_string(index=False))

# one row, as it really looks (pseudonymous ids, shortened for display only)
preview = pool.head(5)[[
    "content_id", "client_id", "content_type", "impressions_90d",
    "impressions_prev_30d", "impressions_last_30d", "avg_position",
    "days_since_last_update", "impressions_lost_30d",
]].copy()
preview["content_id"] = preview["content_id"].str[:14] + "..."
preview["client_id"] = preview["client_id"].str[:12] + "..."
print("\none row = one page x one client x one trailing-90d snapshot:")
preview

rows 30,000 | content_id unique: True | duplicate (client_id, content_id) pairs: 0

                         step  rows  clients
                starter slice 30000       32
           measurable traffic 22006       30
+ flagged declining (my pool) 13152       29

one row = one page x one client x one trailing-90d snapshot:


,content_id,client_id,content_type,impressions_90d,impressions_prev_30d,impressions_last_30d,avg_position,days_since_last_update,impressions_lost_30d
0,content_304f48...,client_f369c...,keyword article,3803,987,578,10.6,20,409
1,content_a1fb4e...,client_4e074...,keyword article,15320,5915,2501,20.3,25,3414
2,content_9aa793...,client_7f225...,keyword article,12581,6089,2382,36.5,20,3707
4,content_d99b7a...,client_3fdba...,keyword article,19140,6452,4211,44.0,14,2241
5,content_d4084a...,client_f369c...,keyword article,3970,1009,617,8.5,20,392


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because there is no single number to put a threshold on, and the right threshold is different for
every client.** Three measurements, below:

1. **No single signal separates.** Ranking by each numeric column on its own (best direction taken for
   free) gives a top AUC of **0.612** (`content_age_days`); everything else is ≤0.57 — barely above the
   0.50 coin flip. An if-statement needs one strong signal to threshold; there isn't one.
2. **The signal flips sign between clients.** That same best-global signal, measured *inside* each
   client, has a median AUC of **0.455** and ranges 0.31 → 0.64. In most accounts it points the
   *opposite* way to the global rule. Combined with per-client base rates from 0.15 to 0.96, any single
   global cutoff is wrong for most clients — a rule would need per-client tuning by hand, forever, on
   32 clients now and 104 in the warehouse.
3. **Missingness is structural, so a rule branches before it compares.** `feedly article` rows have
   **100%** missing keyword data; `keyword article` rows are 28% missing `word_count`. A hand-written
   rule needs a separate branch per content type before any comparison, and a blind `fillna(0)` would
   silently encode content type into the score. A model can carry `has_*` flags and learn the
   interaction instead.

The starter pipeline already shows the gap on this slice under client-holdout validation: rule baseline
precision@50 = **0.240**, random forest = **0.740** — about 12 of the top 50 right versus about 37.

**The honest counter.** That comparison is against the *proxy* label, which is itself a rule, so part of
what the model is beating is a different rule's arithmetic. The real test is the forward-window label in
week 5. If the learned ranking does not beat the transparent rule *there*, the right answer is to ship
the rule and say so — a rule an editor can read is worth more than a model that ties it.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import roc_auc_score

# 1. how far does any ONE column get you?
candidates = ["content_age_days", "search_volume", "days_with_sessions", "ctr", "clicks_90d",
              "scroll_rate", "sessions_90d", "avg_position", "engagement_rate",
              "days_with_impressions", "impressions_90d", "competition", "cpc",
              "days_since_last_update", "word_count", "ai_traffic_pct"]
rows = []
for col in candidates:
    sub = eval_pool[[col, "trend_direction"]].dropna()
    target = (sub["trend_direction"] == "down").astype(int)
    auc = roc_auc_score(target, sub[col])
    rows.append((col, max(auc, 1 - auc), len(sub)))  # best direction, given for free
single = pd.DataFrame(rows, columns=["signal", "auc_best_direction", "rows_used"]).sort_values(
    "auc_best_direction", ascending=False)
print("best any single column can do on its own:")
print(single.head(6).to_string(index=False))

# 2. does the best global signal hold up inside each client?
best = single.iloc[0]["signal"]
per_client_auc = []
for _, g in eval_pool.groupby("client_id"):
    sub = g[[best, "trend_direction"]].dropna()
    target = (sub["trend_direction"] == "down").astype(int)
    if target.nunique() == 2 and len(sub) >= 100:
        per_client_auc.append(roc_auc_score(target, sub[best]))
per_client_auc = pd.Series(per_client_auc)
print(f"\n'{best}' measured INSIDE each client ({len(per_client_auc)} clients with >=100 pages):")
print(f"  min {per_client_auc.min():.2f} | median {per_client_auc.median():.2f} | max {per_client_auc.max():.2f}"
      f"  -> points the other way in {(per_client_auc < 0.5).sum()} of {len(per_client_auc)} clients")

# 3. missingness follows content_type, so a rule needs a branch per type
miss = df.groupby("content_type")[["search_volume", "competition", "word_count"]].apply(
    lambda g: g.isna().mean()).round(2)
miss.insert(0, "pages", df["content_type"].value_counts())
print("\nshare missing, by content type:")
print(miss.to_string())
print("\n-> weak single signals + client-specific direction + type-shaped gaps = too messy for an if-statement")

best any single column can do on its own:
            signal  auc_best_direction  rows_used
  content_age_days            0.611672      22006
     search_volume            0.570291      21441
days_with_sessions            0.555911      22006
               ctr            0.554111      22006
        clicks_90d            0.552681      22006
       scroll_rate            0.551318      21979

'content_age_days' measured INSIDE each client (18 clients with >=100 pages):
  min 0.31 | median 0.45 | max 0.64  -> points the other way in 12 of 18 clients

share missing, by content type:
                    pages  search_volume  competition  word_count
content_type                                                     
comparison article    697           0.00         0.00        0.00
feedly article       2096           1.00         1.00        0.00
keyword article     27207           0.01         0.01        0.28

-> weak single signals + client-specific direction + type-shaped gaps = too messy fo

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.